In [1]:
from copy import deepcopy
import random
import statistics
import seaborn as sns
import json
import pandas as pd
import numpy as np

## DICE

In [2]:
class Dice:
    def __init__(self):
        self.values = [0 for _ in range(5)]
        # self.locked = [False for _ in range(5)]

    def roll(self, lockList = None):

        if lockList:
            locked = lockList
        else:
            locked = []
            
        for i in range(5):
            if i not in locked:
                self.values[i] = random.randint(1,6)

        return self.values

    # def set_locked(self, prompt):

    #     if prompt == [-1]:
    #         for i in range(5):
    #             self.locked[i] = True
        
    #     else:
    #         for i in range(5):
                
    #             if i in prompt:
    #                 self.locked[i] = True
    #             else:
    #                 self.locked[i] = False

    def sum(self):
        return sum(self.values)

    def frequency(self):

        map = [0 for i in range(6)]

        for i in range(5):
            map[self.values[i] - 1] += 1

        frequency = {idx : map[idx - 1] for idx in range(1,7)}
        return frequency

    def sorted(self):
        return sorted(self.values)

    def reset(self):
        self.values = [0 for i in range(5)]
        # self.locked = [False for i in range(5)]

## BOARD

In [3]:
class Board:
    def __init__(self):
        self.scores = [0 for i in range(11)]
        self.taken = [False for i in range(11)]
        self.available_categories = [i for i in range(11)]

    def pick_category(self, idx, score):
        self.scores[idx] = score
        self.taken[idx] = True
        self.available_categories.remove(idx)

    def get_available_categories(self):
        return self.available_categories

## GAMESTATE

In [4]:
class GameState:

    def __init__(self):
        self.dice = Dice()
        self.board = Board()

        self.numTurns = 1
        self.numRolls = 1
        self.total_score = 0

        self.is_game_over = False

    def getSpace(self):
        state = {
            "numTurns" : self.numTurns,
            "numRolls" : self.numRolls,
            "score" : self.total_score,

            "dice" : self.dice.values,

            "scorecard" : self.board.scores,
            "categoriesTaken" : self.board.taken            
        }

        return state

    def reset(self):
        __init__(self)

## ACTION

In [5]:
class Action:

    def __init__(self, type, lockList = None, category = None):

        self.type = type

        if (lockList and category):
            raise ValueError
            
        self.lockList = lockList
        self.category = category

## Environment

In [6]:
def defaultReward(old_state, new_state, action):
    return new_state.total_score - old_state.total_score

In [7]:
class Environment:

    def __init__(self, render_mode = None, gameLogger = None, rewardFunction = defaultReward):

        self.render_mode = render_mode
        self.rewardFunction = rewardFunction
        
        if gameLogger is None:
            gameLogger = GameLogger()

        self.gameLogger = gameLogger
        self.reset()

    def get_observation_space(self):
        return self.state.getSpace()

    def get_action_space(self):
        actions = []

        if self.state.numTurns > 11:
            return actions

        
        if self.state.numRolls <= 2:
            action = Action("roll")
            actions.append(action)

        for idx in self.state.board.get_available_categories():
            action = Action("pick", category = idx)
            actions.append(action)

        return actions

    def reset(self):
        self.state = GameState()
        
        self.state.dice.roll()
        self.gameLogger.log_initial_roll(self.state, self.get_available_scores(), self.state.board.get_available_categories())
        
        self.observation_space = self.get_observation_space()

    def calculate_category_score(self, idx):

        if idx in range(6):
            return self.state.dice.frequency()[idx + 1] * (idx + 1)

        if idx == 6:
            if max(self.state.dice.frequency().values()) >= 3:
                return self.state.dice.sum()
            return 0

        if idx == 7:
            if max(self.state.dice.frequency().values()) >= 4:
                return self.state.dice.sum()
            return 0
            
        if idx == 8:
            if self.state.dice.sorted() == [1,2,3,4,5] or self.state.dice.sorted() == [2,3,4,5,6]:
                return 40
            else:
                return 0
                
        if idx == 9:
            if 2 in self.state.dice.frequency().values() and 3 in self.state.dice.frequency().values():
                return 25
            else:
                return 0
                
        if idx == 10:
            if max(self.state.dice.frequency().values()) == 5:
                return 50
            else:
                return 0
            

    def get_available_scores(self):

        available_scores = [-1 for i in range(11)]
        available_categories = self.state.board.get_available_categories()
        
        for i in available_categories:
            available_scores[i] = self.calculate_category_score(i)

        return available_scores
        

    def step(self, action):

        if (action.type == "roll"):

            old_state = deepcopy(self.state)
            
            self.state.dice.roll(action.lockList)
            self.state.numRolls += 1
            
            self.gameLogger.log_roll(old_state, action, self.state, self.get_available_scores(), old_state.board.get_available_categories())

            self.state.observation_space = self.get_observation_space()
            
            terminated = False

        elif (action.type == "pick"):

            old_state = deepcopy(self.state)

            score = self.get_available_scores()[action.category]

            self.state.board.pick_category(action.category, score)

            self.state.total_score += score
            self.state.numTurns += 1
            
            self.gameLogger.log_pick(old_state, action, self.state, score, self.rewardFunction(old_state, self.state, action), self.state.total_score)

            if len(self.state.board.get_available_categories()) != 0:

                available_scores = self.get_available_scores()
                available_categories = self.state.board.get_available_categories()
                
                self.state.numRolls = 1

                old_state = deepcopy(self.state)
                self.state.dice.roll()

                new_available_scores = self.get_available_scores()
    
                self.gameLogger.log_roll(old_state, action, self.state, new_available_scores, available_categories)

    
            self.state.is_game_over = (
                len(self.state.board.get_available_categories()) == 0
            )

            self.state.observation_space = self.get_observation_space()

            terminated = self.state.is_game_over
            
        reward = self.rewardFunction(old_state, self.state, action)
        
        truncated = False

        info = None

        return self.state, reward, terminated, truncated, info

            
            

## Player

In [8]:
from abc import ABC, abstractmethod

In [9]:
class Player(ABC):

    def __init__(self, name):
        self.name = name

    @abstractmethod
    def act(self, env):
        pass

### RandomPlayer

In [10]:
class RandomPlayer(Player):
    
    def __init__(self):
        super().__init__("Random")

    def act(self, env):
        return random.choice(env.get_action_space())

In [11]:
'''
[
gameNo : i,
score : xxx,
turns: {

        turn : i,

        rolls : [{

                roll : 1 - 3,
                diceBefore : [5,5,5,5,5],

                available_scores : [...],
                available_categories: [...],

                locklist = [...]

                diceAfter : [x,x,x,x,x]

                score obtained: xx
                reward obtained: xx

            
            }],

        categoryPicked = xx
        score_obtained = xx
        reward_obtained
        runningScore = xxx
    
    }
]

'''


'\n[\ngameNo : i,\nscore : xxx,\nturns: {\n\n        turn : i,\n\n        rolls : [{\n\n                roll : 1 - 3,\n                diceBefore : [5,5,5,5,5],\n\n                available_scores : [...],\n                available_categories: [...],\n\n                locklist = [...]\n\n                diceAfter : [x,x,x,x,x]\n\n                score obtained: xx\n                reward obtained: xx\n\n\n            }],\n\n        categoryPicked = xx\n        score_obtained = xx\n        reward_obtained\n        runningScore = xxx\n\n    }\n]\n\n'

## GameLogger

In [24]:
class GameLogger:

    def __init__(self, id = 0, player_type = "player"):

        self.player_type = player_type
        self.id = id
        self.turns = []

        self.total_score = 0
        self.category_scores = [0 for i in range(11)]
        
        self.total_reward = 0

        for i in range(11):
            d = {}
            d["turn"] = i + 1

            d["rolls"] = []

            d["category_picked"] = -1
            d["score_obtained"] = 0
            d["reward_obtained"] = 0
            d["running_score"] = 0

            self.turns.append(d)           

    def log_roll(self, old_state, action, new_state, available_scores, available_categories):
        d = {}
        d["roll"] = new_state.numRolls
        d["dice"] = new_state.dice.values.copy()
        
        d["available_scores"] = available_scores.copy()
        d["available_categories"] = available_categories.copy()

        d["lock_list"] = action.lockList.copy() if action.lockList else None

        self.turns[old_state.numTurns - 1]["rolls"].append(d)

    def log_initial_roll(self, state, available_scores, available_categories):
        d = {}
        d["roll"] = state.numRolls
        d["dice"] = state.dice.values.copy()
        
        d["available_scores"] = available_scores.copy()
        d["available_categories"] = available_categories.copy()

        d["lock_list"] = None


        self.turns[state.numTurns - 1]["rolls"].append(d)

    def log_pick(self, old_state, action, new_state, score_obtained, reward_obtained, running_score):

        d = self.turns[old_state.numTurns - 1]
        d["category_picked"] = action.category
        d["score_obtained"] = score_obtained
        d["reward_obtained"] = reward_obtained
        d["running_score"] = running_score

        self.category_scores[action.category] = score_obtained
        self.total_score += score_obtained
        self.total_reward += reward_obtained

    def get_summary(self):

        summary = {}

        summary["id"] = self.id
        summary["player_type"] = self.player_type
        summary["total_score"] = self.total_score
        summary["total_reward"] = self.total_reward

        for i in range(11):
            summary[f"score_{i}"] = self.category_scores[i]

        return summary
        
        

## Simulation

In [32]:
class Simulation:

    def __init__(self, player, k = 10):

        self.id = 0
        self.player = player
        self.logger = SimulationLogger(k)
        self.scores = []

    def play_game(self):

        env = Environment(gameLogger = GameLogger(id = self.id, player_type = self.player.name))

        while not env.state.is_game_over:
        
            action = self.player.act(env)
            env.step(action)

        self.logger.log_game(env)
        return env.state.total_score

    def run(self, n):
        
        for i in range(n):
            self.scores.append(self.play_game())
            self.id += 1

            
        

## SimulationLogger

In [14]:
class SimulationLogger:

    def __init__(self, k):
        self.k = k
        self.top_k_games = []
        self.bottom_k_games = []

        cols = [

            "id",
            "player_type",
            "total_score",
            "total_reward"
        
        ]

        for i in range(11):
            cols.append(f"score_{i}")
        
        self.summary = pd.DataFrame(columns = cols)

    def log_game(self, env):

        currScore = env.gameLogger.total_score
        
        if len(self.top_k_games) < self.k:
            self.top_k_games.append(deepcopy(env.gameLogger))
            self.top_k_games.sort(key=lambda game: game.total_score, reverse=True)
        
        elif currScore > self.top_k_games[-1].total_score:
            self.top_k_games.append(deepcopy(env.gameLogger))
            self.top_k_games.sort(key=lambda game: game.total_score, reverse=True)
            self.top_k_games = self.top_k_games[:self.k]
        
        
        if len(self.bottom_k_games) < self.k:
            self.bottom_k_games.append(deepcopy(env.gameLogger))
            self.bottom_k_games.sort(key=lambda game: game.total_score)
        
        elif currScore < self.bottom_k_games[-1].total_score:
            self.bottom_k_games.append(deepcopy(env.gameLogger))
            self.bottom_k_games.sort(key=lambda game: game.total_score)
            self.bottom_k_games = self.bottom_k_games[:self.k]
        
        
        self.summary.loc[len(self.summary)] = env.gameLogger.get_summary()

        

        
        

In [33]:
sim = Simulation(RandomPlayer())

sim.run(1000)

In [34]:
df = sim.logger.summary
df.head()

,id,player_type,total_score,total_reward,score_0,score_1,score_2,score_3,score_4,score_5,score_6,score_7,score_8,score_9,score_10
0,0,Random,9,9,2,4,3,0,0,0,0,0,0,0,0
1,1,Random,19,19,1,4,0,4,10,0,0,0,0,0,0
2,2,Random,15,15,1,0,3,0,5,6,0,0,0,0,0
3,3,Random,27,27,0,0,3,0,5,0,19,0,0,0,0
4,4,Random,6,6,0,0,0,0,0,6,0,0,0,0,0


In [35]:
[sim.logger.top_k_games[i].total_score for i in range(10)]

[118, 91, 84, 82, 81, 78, 76, 75, 72, 70]

In [36]:
[sim.logger.bottom_k_games[i].total_score for i in range(10)]

[0, 0, 0, 0, 1, 1, 2, 2, 2, 2]

In [41]:
sim.logger.top_k_games[0].turns

[{'turn': 1,
  'rolls': [{'roll': 1,
    'dice': [4, 2, 4, 4, 2],
    'available_scores': [0, 4, 0, 12, 0, 0, 16, 0, 0, 25, 0],
    'available_categories': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'lock_list': None}],
  'category_picked': 3,
  'score_obtained': 12,
  'reward_obtained': 12,
  'running_score': 12},
 {'turn': 2,
  'rolls': [{'roll': 1,
    'dice': [4, 6, 2, 5, 2],
    'available_scores': [0, 4, 0, -1, 5, 6, 0, 0, 0, 0, 0],
    'available_categories': [0, 1, 2, 4, 5, 6, 7, 8, 9, 10],
    'lock_list': None}],
  'category_picked': 5,
  'score_obtained': 6,
  'reward_obtained': 6,
  'running_score': 18},
 {'turn': 3,
  'rolls': [{'roll': 1,
    'dice': [2, 4, 5, 1, 1],
    'available_scores': [2, 2, 0, -1, 5, -1, 0, 0, 0, 0, 0],
    'available_categories': [0, 1, 2, 4, 6, 7, 8, 9, 10],
    'lock_list': None},
   {'roll': 2,
    'dice': [6, 6, 6, 2, 2],
    'available_scores': [0, 4, 0, -1, 0, -1, 22, 0, 0, 25, 0],
    'available_categories': [0, 1, 2, 4, 6, 7, 8, 9, 10],
    '

In [39]:
max(sim.scores)

118

In [40]:
np.mean(sim.scores)

np.float64(23.815)